In [28]:
#Ref
import pandas as pd
import glob
import numpy as np
import plotly.express as px
from spirit import simulation, state,quantities, hamiltonian,parameters,geometry,configuration,system,io
import numpy as np
import os
import matplotlib.pyplot as plt
import multiprocessing as mp
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
from scipy.stats import linregress
from tqdm import tqdm
import time
from datetime import timedelta
from collections import defaultdict

path = '/Users/jiakai/Desktop/SURF/code/_spirit/'
n_cycles = 120


# Pattern to match all relevant CSV files
file_pattern = path + 'MCS_decay_gammas_dim_10_with_SEM_errorbars_Ht2.3*.csv'

# List all matching files
csv_files = glob.glob(file_pattern)

# Print how many files were found
print(f"Found {len(csv_files)} files.")
repeats = len(csv_files)

for f in csv_files:
    print(f)

# Read and concatenate all files
df_all = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)

Found 2 files.
/Users/jiakai/Desktop/SURF/code/_spirit/MCS_decay_gammas_dim_10_with_SEM_errorbars_Ht2.3_1.csv
/Users/jiakai/Desktop/SURF/code/_spirit/MCS_decay_gammas_dim_10_with_SEM_errorbars_Ht2.3.csv


In [29]:
grouped = df_all.groupby(['gamma', 'i']).agg(
        chi_mean=('chi', 'mean'),
        chi_std=('chi', 'std')
    ).reset_index()

grouped['chi_sem'] = grouped['chi_std'] / ((88+176) ** 0.5)

fig = go.Figure()

unique_gammas = grouped['gamma'].unique()
for gamma in unique_gammas:
    df_gamma = grouped[grouped['gamma'] == gamma]

    fig.add_trace(go.Scatter(
        x=df_gamma['i'],
        y=df_gamma['chi_mean'],
        mode='lines+markers',
        name=f"γ = {gamma:.1e}",
        error_y=dict(
            type='data',
            array=df_gamma['chi_sem'],
            visible=True
        )
    ))

fig.update_layout(
    title=f"Average Susceptibility χ vs i for different γ, relaxed at Ht = {2.3}T",
    xaxis_title="MCS (step index i)",
    yaxis_title="χ (chi)",
    legend_title="Tunneling γ",
    template="plotly_white",
    width=900,
    height=600
)

fig.write_html(f"MCS_decay_gammas_dim_{10}_with_SEM_errorbars_Ht{2.3}_combined.html")
fig_ln = go.Figure()

for gamma in unique_gammas:
    df_gamma = grouped[grouped['gamma'] == gamma]

    # Avoid log of zero or negative numbers
    df_gamma = df_gamma[df_gamma['chi_mean'] > 0]

    fig_ln.add_trace(go.Scatter(
        x=df_gamma['i'],
        y=np.log(df_gamma['chi_mean']),
        mode='lines+markers',
        name=f"γ = {gamma:.1e}",
        error_y=dict(
            type='data',
            array=df_gamma['chi_sem'] / df_gamma['chi_mean'],  # Propagation of error in log(chi)
            visible=True
        )
    ))

fig_ln.update_layout(
    title=f"ln(χ) vs i for different γ, relaxed at Ht = {2.3}T",
    xaxis_title="MCS (step index i)",
    yaxis_title="ln(χ)",
    legend_title="Tunneling γ",
    template="plotly_white",
    width=900,
    height=600
)

fig_ln.write_html(f"ln_MCS_decay_gammas_dim_{10}_with_SEM_errorbars_Ht{2.3}_combined.html")

fig_lnln = go.Figure()

for gamma in unique_gammas:
    df_gamma = grouped[grouped['gamma'] == gamma].copy()

    # Filter out invalid values
    df_gamma = df_gamma[(df_gamma['chi_mean'] > 0) & (np.log(df_gamma['chi_mean']) < 0)]

    # Compute x = ln(i), y = ln(-ln(chi))
    df_gamma['ln_i'] = np.log(df_gamma['i'])
    df_gamma['ln_ln_chi'] = np.log(-np.log(df_gamma['chi_mean']))

    # Error propagation
    df_gamma['ln_ln_chi_sem'] = (
            np.abs(1 / (np.log(df_gamma['chi_mean']) * df_gamma['chi_mean'])) * df_gamma['chi_sem']
    )

    fig_lnln.add_trace(go.Scatter(
        x=df_gamma['ln_i'],
        y=df_gamma['ln_ln_chi'],
        mode='lines+markers',
        name=f"γ = {gamma:.1e}",
        error_y=dict(
            type='data',
            array=df_gamma['ln_ln_chi_sem'],
            visible=True
        )
    ))

fig_lnln.update_layout(
    title=f"ln(-ln(χ)) vs ln(i) for different γ, relaxed at Ht = {2.3}T",
    xaxis_title="ln(MCS step index i)",
    yaxis_title="ln(-ln(χ))",
    legend_title="Tunneling γ",
    template="plotly_white",
    width=900,
    height=600
)

fig_lnln.write_html(f"lnln_MCS_decay_gammas_dim_{10}_with_SEM_errorbars_Ht{2.3}_combined.html")


/Users/jiakai/Desktop/SURF/code/venv/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning:

divide by zero encountered in log



In [30]:

# Filter out inf or NaN
df_fit = df_gamma.replace([np.inf, -np.inf], np.nan).dropna(subset=['ln_i', 'ln_ln_chi'])

# Keep only rows where ln(i) ≥ 2
df_fit = df_fit[df_fit['ln_i'] >= 3]

# Linear fit: ln_ln_chi = a * ln_i + b
coeffs = np.polyfit(df_fit['ln_i'], df_fit['ln_ln_chi'], 1)
a, b = coeffs

# Generate fit line
x_fit = np.linspace(df_fit['ln_i'].min(), 6, 200)
y_fit = a * x_fit + b

# Add best fit line to the plot
fig_lnln.add_trace(go.Scatter(
    x=x_fit,
    y=y_fit,
    mode='lines',
    name=f"Fit: y = {a:.6f}x + {b:.6f}",
    line=dict(dash='dash', color='black')
))

fig_lnln.show()

In [26]:
print(np.log(200), np.log(400))

5.298317366548036 5.991464547107982


In [27]:
print(np.exp(-np.exp(a*np.log(200) + b)))
print(np.exp(-np.exp(a*np.log(400) + b)))

0.5061166544130341
0.4975409580395484
